In [9]:
# ==========================================================
# Global Random Forest using MLForecast (Nixtla)
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Nixtla
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import Differences, LocalBoxCox


# Machine Learning
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [10]:
processed_dir = Path("data/processed")
raw_dir = Path("data/raw")

sales = pd.read_csv(
    processed_dir / "sales_clean.csv",
    parse_dates=["date"],
    low_memory=False
)

future = pd.read_csv(
    processed_dir / "future_clean.csv",
    parse_dates=["date"],
    low_memory=False
)



In [11]:
# ==========================================================
# Remove unavailable feature
# ==========================================================

sales = sales.drop(columns=["customers"])
future = future.drop(columns=["customers"])

In [12]:
# ==========================================================
# Sort
# ==========================================================

sales = sales.sort_values(
    ["store_id", "date"]
).reset_index(drop=True)

future = future.sort_values(
    ["store_id", "date"]
).reset_index(drop=True)

In [13]:
# ==========================================================
# Rename for MLForecast
# ==========================================================

sales = sales.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds",
        "sales": "y"
    }
)

future = future.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds"
    }
)
# ==========================================================
# Rename Columns for MLForecast
# ==========================================================

sales = sales.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds",
        "sales": "y"
    }
)

future = future.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds"
    }
)

# ==========================================================
# Encode Categorical Features
# ==========================================================

state_holiday_map = {
    "0": 0,
    "a": 1,
    "b": 2,
    "c": 3,
    0: 0,
    0.0: 0
}

store_type_map = {
    "a": 0,
    "b": 1,
    "c": 2,
    "d": 3
}

assortment_map = {
    "a": 0,
    "b": 1,
    "c": 2
}

for df in [sales, future]:

    df["state_holiday"] = (
        df["state_holiday"]
        .replace(state_holiday_map)
        .astype(int)
    )

    df["store_type"] = (
        df["store_type"]
        .map(store_type_map)
        .astype(int)
    )

    df["assortment"] = (
        df["assortment"]
        .map(assortment_map)
        .astype(int)
    )

In [14]:
# ==========================================================
# Create Additional Dynamic Features
# ==========================================================
# These features are known in advance and can be used as
# dynamic covariates during forecasting.
# ==========================================================

# Weekend indicator
sales["is_weekend"] = (
    sales["ds"].dt.dayofweek >= 5
).astype(int)

future["is_weekend"] = (
    future["ds"].dt.dayofweek >= 5
).astype(int)

# Quarter of the year
sales["quarter"] = sales["ds"].dt.quarter

future["quarter"] = future["ds"].dt.quarter

# ISO week number
sales["week_of_year"] = (
    sales["ds"]
    .dt.isocalendar()
    .week
    .astype(int)
)

future["week_of_year"] = (
    future["ds"]
    .dt.isocalendar()
    .week
    .astype(int)
)

# Promotion during weekends
sales["promo_weekend"] = (
    sales["promo"] *
    sales["is_weekend"]
)

future["promo_weekend"] = (
    future["promo"] *
    future["is_weekend"]
)

print("=" * 60)
print("Additional Features")
print("=" * 60)

print(sales[
    [
        "unique_id",
        "ds",
        "promo",
        "is_weekend",
        "quarter",
        "week_of_year",
        "promo_weekend"
    ]
].head())

Additional Features
  unique_id         ds  promo  is_weekend  quarter  week_of_year  \
0   store_1 2013-01-07      1           0        1             2   
1   store_1 2013-01-08      1           0        1             2   
2   store_1 2013-01-09      1           0        1             2   
3   store_1 2013-01-10      1           0        1             2   
4   store_1 2013-01-11      1           0        1             2   

   promo_weekend  
0              0  
1              0  
2              0  
3              0  
4              0  


In [15]:
# ==========================================================
# Hyperparameter Grid
# ==========================================================

parameter_grid = [

    {
        "name": "RF200_SQRT",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF300_SQRT_Leaf5",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    }

]

# ==========================================================
# Hyperparameter Tuning using Rolling Forecast Origin CV
# ==========================================================

results = []

for params in parameter_grid:
    print("=" * 60)
    print(params["name"])
    print("=" * 60)

    # ------------------------------------------------------
    # Random Forest
    # ------------------------------------------------------

    model = RandomForestRegressor(

        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=42,
        n_jobs=-1

    )

    # ------------------------------------------------------
    # MLForecast
    # ------------------------------------------------------

    fcst = MLForecast(

        models=[model],

        freq="D",

        lags=[
            1,
            7,
            14,
            21,
            28
        ],

        lag_transforms={

            7: [
                RollingMean(window_size=4)
            ],

            28: [
                RollingMean(window_size=2)
            ]

        },

        date_features=[
            "dayofweek",
            "month"
        ]

    )

    # ------------------------------------------------------
    # Rolling Forecast Origin Cross Validation
    # ------------------------------------------------------

    cv = fcst.cross_validation(

        df=sales,

        h=42,

        n_windows=3,

        refit=5,

        static_features=[]

    )

    prediction_col = cv.columns[-1]

    valid = cv["y"] > 0

    mae = mean_absolute_error(
        cv.loc[valid, "y"],
        cv.loc[valid, prediction_col]
    )

    rmse = np.sqrt(
        mean_squared_error(
            cv.loc[valid, "y"],
            cv.loc[valid, prediction_col]
        )
    )

    mape = (
               np.abs(
                   (
                           cv.loc[valid, "y"]
                           - cv.loc[valid, prediction_col]
                   )
                   / cv.loc[valid, "y"]
               )
           ).mean() * 100

    results.append({

        "Model": params["name"],
        "Trees": params["n_estimators"],
        "Leaf": params["min_samples_leaf"],
        "Max Features": params["max_features"],
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape

    })

    print(f"MAE :  {mae:.2f}")
    print(f"RMSE:  {rmse:.2f}")
    print(f"MAPE:  {mape:.2f}%")

# ==========================================================
# Hyperparameter Tuning Results
# ==========================================================

results = pd.DataFrame(results)

results = results.sort_values(
    by="MAPE",
    ascending=True
)

print(results)

results.to_csv(
    processed_dir / "rf_cv_results.csv",
    index=False
)

# ==========================================================
# Select Best Hyperparameters
# ==========================================================

best_result = results.iloc[0]

best_model = RandomForestRegressor(

    n_estimators=int(best_result["Trees"]),

    min_samples_leaf=int(best_result["Leaf"]),

    max_features=best_result["Max Features"],

    random_state=42,

    n_jobs=-1

)

# ==========================================================
# Train Final Model
# ==========================================================

final_fcst = MLForecast(

    models=[best_model],

    freq="D",

    lags=[
        1,
        7,
        14,
        21,
        28
    ],

    lag_transforms={

        7: [
            RollingMean(window_size=4)
        ],

        28: [
            RollingMean(window_size=2)
        ]

    },

    date_features=[
        "dayofweek",
        "month"
    ]

)

final_fcst.fit(

    sales,

    static_features=[]

)

print("=" * 60)
print("Best Model")
print("=" * 60)
print(best_result)
print()
print("Final model trained successfully.")

RF200_SQRT
MAE :  726.27
RMSE:  1063.57
MAPE:  10.12%
RF300_SQRT_Leaf5
MAE :  728.59
RMSE:  1076.46
MAPE:  10.15%
              Model  Trees  Leaf Max Features         MAE         RMSE  \
0        RF200_SQRT    200     1         sqrt  726.266500  1063.570237   
1  RF300_SQRT_Leaf5    300     5         sqrt  728.586173  1076.461454   

        MAPE  
0  10.118750  
1  10.154861  
Best Model
Model            RF200_SQRT
Trees                   200
Leaf                      1
Max Features           sqrt
MAE                726.2665
RMSE            1063.570237
MAPE               10.11875
Name: 0, dtype: object

Final model trained successfully.


In [16]:
# ==========================================================
# Generate Features for the Final Model
# ==========================================================

features = final_fcst.preprocess(

    sales,

    static_features=[]

)

print("=" * 60)
print("Generated Features")
print("=" * 60)

print(features.head())

print(features.shape)

# ==========================================================
# Create Evaluation Dataset
# ==========================================================

eval_size = 42 * sales["unique_id"].nunique()

train_features = features.iloc[:-eval_size].copy()

test_features = features.iloc[-eval_size:].copy()

# Randomly sample observations to speed up permutation importance
test_features = test_features.sample(

    n=min(5000, len(test_features)),

    random_state=42

)

print("=" * 60)
print("Train Features:", train_features.shape)
print("Test Features :", test_features.shape)

Generated Features
   unique_id         ds       y  open  promo  state_holiday  school_holiday  \
29   store_1 2013-02-05  6049.0     1      1              0               0   
30   store_1 2013-02-06  6140.0     1      1              0               0   
31   store_1 2013-02-07  5499.0     1      1              0               0   
32   store_1 2013-02-08  5681.0     1      1              0               0   
33   store_1 2013-02-09  5370.0     1      0              0               0   

    store_type  assortment  competition_distance  ...  promo_weekend    lag1  \
29           2           0                1270.0  ...              0  7032.0   
30           2           0                1270.0  ...              0  6049.0   
31           2           0                1270.0  ...              0  6140.0   
32           2           0                1270.0  ...              0  5499.0   
33           2           0                1270.0  ...              0  5681.0   

      lag7   lag14   lag2